In [ ]:
### Machine Specifications: Local Windows Environment (Processor: 12th Gen Intel(R) Core(TM) i5-12400F @ 2.50 GHz, RAM: 16.0 GB).
### Timing Mechanism: Python's built-in `time.time()` module (measuring wall-clock time in seconds).
### Number of Repetitions: Each algorithm is executed 3 times per test case using randomly selected starting nodes to account for system noise and caching.
### Reported Times: The final reported execution times are the average (mean) across the 3 runs.
### Input Consistency: To ensure a fair and scientifically valid comparison, both Dijkstra's and Bellman-Ford's algorithms are executed on the exact same graph subset, edges, weights, and source/target nodes during every iteration.

In [ ]:
import heapq
import time
import os
import numpy as np
import pandas as pd
import kagglehub

# Download and Load the dataset into Pandas
path = kagglehub.dataset_download("wolfram77/graphs-snap-roadnet")
df_path = os.path.join(path, 'roadNet-CA.mtx')
df = pd.read_csv(df_path, sep=r'\s+', header=None, comment='%')

# Clean the data
edges_df = df.iloc[1:, 0:2].copy()
edges_df.columns = ['Source', 'Target']
edges_df['Source'] = edges_df['Source'].astype(int)
edges_df['Target'] = edges_df['Target'].astype(int)

#Add random weights (1 to 50)
np.random.seed(42)
edges_df['Weight'] = np.random.randint(1, 50, size=len(edges_df))

print(f"Dataset successfully prepared! Total available edges: {len(edges_df):,}")

Using Colab cache for faster access to the 'graphs-snap-roadnet' dataset.
Dataset successfully prepared! Total available edges: 2,766,607


In [ ]:
def build_graph(df_subset):
    """Converts the pandas dataframe into a dictionary-based graph."""
    graph = {}
    for _, row in df_subset.iterrows():
        u, v, w = int(row['Source']), int(row['Target']), int(row['Weight'])
        if u not in graph: graph[u] = {}
        if v not in graph: graph[v] = {}

        graph[u][v] = w
        graph[v][u] = w
    return graph

def dijkstra(graph, source):
    """Greedy Approach: Dijkstra's Algorithm"""
    distances = {node: float('inf') for node in graph}
    distances[source] = 0
    pq = [(0, source)]

    while pq:
        current_distance, current_node = heapq.heappop(pq)

        if current_distance > distances[current_node]:
            continue

        for neighbor, weight in graph[current_node].items():
            distance = current_distance + weight
            if distance < distances[neighbor]:
                distances[neighbor] = distance
                heapq.heappush(pq, (distance, neighbor))
    return distances

def bellman_ford(graph, source):
    """Dynamic Programming Approach: Bellman-Ford Algorithm"""
    distances = {node: float('inf') for node in graph}
    distances[source] = 0

    edges = [(u, v, w) for u in graph for v, w in graph[u].items()]
    V = len(graph)


    for _ in range(V - 1):
        for u, v, w in edges:
            if distances[u] != float('inf') and distances[u] + w < distances[v]:
                distances[v] = distances[u] + w


    for u, v, w in edges:
        if distances[u] != float('inf') and distances[u] + w < distances[v]:
            return None
    return distances


results_log = []

In [ ]:
import random

runs_per_case = 3

cases = {
    "Best Case (500 Edges)": edges_df.head(500),
    "Average Case (2,500 Edges)": edges_df.head(2500),
    "Worst Case (10,000 Edges)": edges_df.head(10000)
}

results_log = []

for case_name, subset in cases.items():
    print(f"\n--- Executing {case_name} ---")
    graph = build_graph(subset)
    nodes = list(graph.keys())
    print(f"Network Size: {len(graph)} Nodes")

    d_times = []
    bf_times = []

    for i in range(runs_per_case):

        source_node = random.choice(nodes)

        #  Dijkstra
        start_time = time.time()
        dist_dijkstra = dijkstra(graph, source_node)
        d_times.append(time.time() - start_time)

        # Bellman-Ford
        start_time = time.time()
        dist_bf = bellman_ford(graph, source_node)
        bf_times.append(time.time() - start_time)

        # target
        reachable_nodes = [n for n, d in dist_dijkstra.items() if 0 < d < float('inf')]

        if reachable_nodes:
            target_node = random.choice(reachable_nodes)
            print(f"  Run {i+1} | Source: {source_node} -> Target: {target_node} | Path Cost -> Dijkstra: {dist_dijkstra[target_node]} | Bellman: {dist_bf[target_node]}")
        else:
            print(f"  Run {i+1} | Source: {source_node} is totally isolated in this subgraph. No paths found.")

    # Averages
    avg_d_time = sum(d_times) / runs_per_case
    avg_bf_time = sum(bf_times) / runs_per_case

    print(f"  >> Average Dijkstra Time:     {avg_d_time:.6f} seconds")
    print(f"  >> Average Bellman-Ford Time: {avg_bf_time:.6f} seconds")

    results_log.append({
        "Case": case_name,
        "Avg_Dijkstra_Time": avg_d_time,
        "Avg_Bellman_Time": avg_bf_time
    })


--- Executing Best Case (500 Edges) ---
Network Size: 419 Nodes
  Run 1 | Source: 219 -> Target: 220 | Path Cost -> Dijkstra: 8 | Bellman: 8
  Run 2 | Source: 223 -> Target: 7919 | Path Cost -> Dijkstra: 49 | Bellman: 49
  Run 3 | Source: 1639780 -> Target: 36 | Path Cost -> Dijkstra: 143 | Bellman: 143
  >> Average Dijkstra Time:     0.000153 seconds
  >> Average Bellman-Ford Time: 0.079731 seconds

--- Executing Average Case (2,500 Edges) ---
Network Size: 2016 Nodes
  Run 1 | Source: 34 -> Target: 563 | Path Cost -> Dijkstra: 927 | Bellman: 927
  Run 2 | Source: 531 -> Target: 5422 | Path Cost -> Dijkstra: 82 | Bellman: 82
  Run 3 | Source: 1088 -> Target: 888 | Path Cost -> Dijkstra: 703 | Bellman: 703
  >> Average Dijkstra Time:     0.001681 seconds
  >> Average Bellman-Ford Time: 2.864222 seconds

--- Executing Worst Case (10,000 Edges) ---
Network Size: 7236 Nodes
  Run 1 | Source: 719 -> Target: 694 | Path Cost -> Dijkstra: 133 | Bellman: 133
  Run 2 | Source: 36982 -> Target:

In [ ]:
results_df = pd.DataFrame(results_log)
results_df['Time_Diff (Sec)'] = results_df['Avg_Bellman_Time'] - results_df['Avg_Dijkstra_Time']
results_df['Bellman_is_Slower_By'] = (results_df['Avg_Bellman_Time'] / results_df['Avg_Dijkstra_Time']).round(2).astype(str) + "x"

print("\n--- Final Performance Analysis (Averaged over 3 runs) ---")
display(results_df)


--- Final Performance Analysis (Averaged over 3 runs) ---


,Case,Avg_Dijkstra_Time,Avg_Bellman_Time,Time_Diff (Sec),Bellman_is_Slower_By
0,Best Case (500 Edges),0.000153,0.079731,0.079578,520.63x
1,"Average Case (2,500 Edges)",0.001681,2.864222,2.862541,1703.47x
2,"Worst Case (10,000 Edges)",0.002593,31.462333,31.459740,12133.38x
